# Week 7 - Delta Lake ![Assignment](path)

## Objective

In this assignment, I implemented an incremental data processing workflow using Delta Lake on the Superstore dataset. The workflow covers data loading, cleaning, Delta table creation, simulation of incremental data, MERGE operations, and validation of the final output.

### Technologies Used
- Databricks Free Edition
- Apache Spark (PySpark)
- Delta Lake

## Step 1 : Load Dataset

The Superstore dataset is loaded from Unity Catalog into a Spark DataFrame. This dataset will be used throughout the assignment for performing data cleaning and Delta Lake operations.

In [0]:
# Load the uploaded Superstore table
df = spark.table("workspace.default.sample_superstore")
display(df.limit(10))

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47


### My Observation

The dataset loaded without any issues. From the preview, I can see that it contains order details, customer information, product details, sales, discount and profit, making it suitable for demonstrating Delta Lake operations.

## Step 2 : Explore Dataset

Basic information such as row count, column count and schema is checked before performing any transformations.

In [0]:
# Display basic informastion
print("Number of rows :", df.count())
print("Number of columns :", len(df.columns))

# Display schema
df.printSchema()

Number of rows : 9994
Number of columns : 21
root
 |-- Row ID: long (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: long (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



### My Observation

The dataset contains around 10,000 records with 21 attributes. Before performing any updates, checking the schema and row count helped me understand the dataset structure.

## Step 3 : Data Cleaning

Before creating the Delta table, I performed some basic data cleaning. First, I checked the dataset for missing values. Then I removed duplicate records and renamed the column names by replacing spaces with underscores, since Delta Lake does not support spaces in column names.

In [0]:
# Check for null values
from pyspark.sql.functions import col, sum

null_values = df.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]
)
display(null_values)

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Remove duplicate records
clean_df = df.dropDuplicates()

print("Rows before cleaning :", df.count())
print("Rows after cleaning :", clean_df.count())

Rows before cleaning : 9994
Rows after cleaning : 9994


In [0]:
# Rename columns by replacing spaces with underscores
clean_df = clean_df.toDF(*[
    c.replace(" ", "_")
     .replace("-", "_")
     .replace("/", "_")
    for c in clean_df.columns
])

# Check the new column names
print(clean_df.columns)

['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


### My Observation

While cleaning the dataset, I found that duplicate records did not significantly affect the data. However, renaming the columns was necessary because Delta Lake does not allow spaces or special characters in column names. This step helped avoid errors while creating the Delta table.

## Step 4 : Create Delta Table

After cleaning the dataset, I stored it as a Delta table. Using Delta format allows updates, inserts, and MERGE operations while maintaining data consistency through ACID transactions.

In [0]:
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.superstore_delta")

delta_df = spark.table("workspace.default.superstore_delta")

display(delta_df.limit(10))

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
17,CA-2014-105893,2014-11-11,2014-11-18,Standard Class,PK-19075,Pete Kriz,Consumer,United States,Madison,Wisconsin,53711,Central,OFF-ST-10004186,Office Supplies,Storage,"Stur-D-Stor Shelving, Vertical 5-Shelf: 72""H x 36""W x 18 1/2""D",665.88,6,0.0,13.3176
79,US-2014-147606,2014-11-26,2014-12-01,Second Class,JE-15745,Joel Eaton,Consumer,United States,Houston,Texas,77070,Central,FUR-FU-10003194,Furniture,Furnishings,"Eldon Expressions Desk Accessory, Wood Pencil Holder, Oak",19.3,5,0.6,-14.475
110,CA-2015-129476,2015-10-15,2015-10-20,Standard Class,PA-19060,Pete Armstrong,Home Office,United States,Orland Park,Illinois,60462,Central,TEC-AC-10000844,Technology,Accessories,Logitech�Gaming G510s - Keyboard,339.96,5,0.2,67.992
118,CA-2015-110457,2015-03-02,2015-03-06,Standard Class,DK-13090,Dave Kipp,Consumer,United States,Seattle,Washington,98103,West,FUR-TA-10001768,Furniture,Tables,Hon Racetrack Conference Tables,787.53,3,0.0,165.3813
135,CA-2016-145583,2016-10-13,2016-10-19,Standard Class,LC-16885,Lena Creighton,Consumer,United States,Roseville,California,95661,West,OFF-PA-10001736,Office Supplies,Paper,Xerox 1880,35.44,1,0.0,16.6568
138,CA-2016-145583,2016-10-13,2016-10-19,Standard Class,LC-16885,Lena Creighton,Consumer,United States,Roseville,California,95661,West,OFF-BI-10004781,Office Supplies,Binders,GBC Wire Binding Strips,76.176,3,0.2,26.6616
140,CA-2016-145583,2016-10-13,2016-10-19,Standard Class,LC-16885,Lena Creighton,Consumer,United States,Roseville,California,95661,West,FUR-FU-10001706,Furniture,Furnishings,Longer-Life Soft White Bulbs,43.12,14,0.0,20.6976
148,CA-2016-114489,2016-12-05,2016-12-09,Standard Class,JE-16165,Justin Ellison,Corporate,United States,Franklin,Wisconsin,53132,Central,TEC-PH-10000215,Technology,Phones,Plantronics Cordless�Phone Headset�with In-line Volume - M214C,384.45,11,0.0,103.8015
153,CA-2016-158834,2016-03-13,2016-03-16,First Class,TW-21025,Tamara Willingham,Home Office,United States,Scottsdale,Arizona,85254,West,TEC-PH-10001254,Technology,Phones,Jabra BIZ 2300 Duo QD Duo Corded�Headset,203.184,2,0.2,15.2388
175,US-2014-100853,2014-09-14,2014-09-19,Standard Class,JB-15400,Jennifer Braxton,Corporate,United States,Chicago,Illinois,60623,Central,OFF-AP-10000891,Office Supplies,Appliances,Kensington 7 Outlet MasterPiece HOMEOFFICE Power Control Center,52.448,2,0.8,-131.12


### My Observation

The Delta table was created successfully. Unlike a normal Parquet table, this Delta table can now be updated using the MERGE operation, which is one of the main advantages of Delta Lake.

In [0]:
display(
    delta_df.groupBy("Category")
            .sum("Sales")
            .orderBy("sum(Sales)", ascending=False)
)

Category,sum(Sales)
Technology,836154.032999998
Furniture,741999.7952999991
Office Supplies,719047.032000002


### Additional Analysis

I explored the sales distribution across different product categories. Technology and Furniture contribute a significant portion of total sales, showing that the dataset has a balanced distribution for demonstrating data engineering operations.

## Step 5 : Simulate Incremental Data

In real-world data engineering projects, new data arrives continuously instead of replacing the entire dataset. To simulate this scenario, I created a small incremental dataset using a few existing records, updated some values, and added a completely new record.

In [0]:
# Create incremental dataset from existing records

incremental_df = clean_df.limit(5)

display(incremental_df)

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164


### My Observation

Instead of creating a completely new dataset, I selected a small subset from the existing data. This better represents how incremental batches usually arrive in production systems.

In [0]:
from pyspark.sql.functions import when, col

incremental_df = incremental_df.withColumn(
    "Sales",
    when(col("Row_ID") == 1, 9999.99)
    .otherwise(col("Sales"))
)

incremental_df = incremental_df.withColumn(
    "Profit",
    when(col("Row_ID") == 2, 1200.00)
    .otherwise(col("Profit"))
)

display(incremental_df)

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,9999.99,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,1200.0
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164


### My Observation

I updated the Sales and Profit values for two existing records. These modifications simulate updates received from a source system, such as corrected transaction values.

### Step 5.1 : Add New Record

To simulate newly arriving data, I added one new customer order to the incremental dataset. This record does not exist in the original Delta table and should be inserted during the MERGE operation.


In [0]:
from pyspark.sql import Row

new_record = spark.createDataFrame([
    Row(
        Row_ID=10001,
        Order_ID="CA-2026-999999",
        Order_Date="2026-07-06",
        Ship_Date="2026-07-08",
        Ship_Mode="Second Class",
        Customer_ID="CG-99999",
        Customer_Name="Sanjana",
        Segment="Consumer",
        Country="India",
        City="Delhi",
        State="Delhi",
        Postal_Code=110001,
        Region="North",
        Product_ID="TEC-PH-99999",
        Category="Technology",
        Sub_Category="Phones",
        Product_Name="Smart Phone",
        Sales=25000.0,
        Quantity=2,
        Discount=0.0,
        Profit=5000.0
    )
])

incremental_df = incremental_df.unionByName(new_record)

display(incremental_df)

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
17,CA-2014-105893,2014-11-11,2014-11-18,Standard Class,PK-19075,Pete Kriz,Consumer,United States,Madison,Wisconsin,53711,Central,OFF-ST-10004186,Office Supplies,Storage,"Stur-D-Stor Shelving, Vertical 5-Shelf: 72""H x 36""W x 18 1/2""D",665.88,6,0.0,13.3176
79,US-2014-147606,2014-11-26,2014-12-01,Second Class,JE-15745,Joel Eaton,Consumer,United States,Houston,Texas,77070,Central,FUR-FU-10003194,Furniture,Furnishings,"Eldon Expressions Desk Accessory, Wood Pencil Holder, Oak",19.3,5,0.6,-14.475
110,CA-2015-129476,2015-10-15,2015-10-20,Standard Class,PA-19060,Pete Armstrong,Home Office,United States,Orland Park,Illinois,60462,Central,TEC-AC-10000844,Technology,Accessories,Logitech�Gaming G510s - Keyboard,339.96,5,0.2,67.992
118,CA-2015-110457,2015-03-02,2015-03-06,Standard Class,DK-13090,Dave Kipp,Consumer,United States,Seattle,Washington,98103,West,FUR-TA-10001768,Furniture,Tables,Hon Racetrack Conference Tables,787.53,3,0.0,165.3813
135,CA-2016-145583,2016-10-13,2016-10-19,Standard Class,LC-16885,Lena Creighton,Consumer,United States,Roseville,California,95661,West,OFF-PA-10001736,Office Supplies,Paper,Xerox 1880,35.44,1,0.0,16.6568
10001,CA-2026-999999,2026-07-06,2026-07-08,Second Class,CG-99999,Sanjana,Consumer,India,Delhi,Delhi,110001,North,TEC-PH-99999,Technology,Phones,Smart Phone,25000.0,2,0.0,5000.0


### My Observation

The incremental dataset now contains both updated records and one completely new record. This setup closely represents real-world scenarios where incoming data includes a mix of updates and new entries.

In [0]:
# print("=" * 50)
print("Incremental Dataset Summary")
# print("=" * 50)

print("Records in Incremental Dataset :", incremental_df.count())
print("New Record Added : 1")
print("Updated Existing Records : 2")

Incremental Dataset Summary
Records in Incremental Dataset : 6
New Record Added : 1
Updated Existing Records : 2


### Key Finding

The incremental dataset contains both updated records and a newly added record. This combination helps demonstrate how the Delta MERGE operation handles updates and inserts in a single transaction.

In [0]:
display(
    incremental_df.groupBy("Category")
                  .count()
)

Category,count
Office Supplies,2
Technology,2
Furniture,2


### Additional Analysis

I briefly explored the distribution of records in the incremental dataset. Since the sample is small, this check helps verify that the data was created as expected before applying the MERGE operation.

## Step 6 : Apply MERGE Operation

The incremental dataset is merged into the Delta table using the `MERGE` operation. Existing records are updated based on the `Row_ID`, while records that do not exist in the target table are inserted as new records.

In [0]:
from delta.tables import DeltaTable

# Access Delta Table
delta_table = DeltaTable.forName(
    spark,
    "workspace.default.superstore_delta"
)

# Perform MERGE operation
(
    delta_table.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.Row_ID = source.Row_ID"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE operation completed successfully.")

MERGE operation completed successfully.


### My Observation

The MERGE operation updated the existing records and inserted the new record into the Delta table. This demonstrates how Delta Lake efficiently handles incremental data processing without replacing the complete dataset.


## Step 7 : Validate Final Output

After completing the MERGE operation, I validated the results by displaying the final Delta table, checking the total number of records, and verifying that no duplicate `Row_ID` values were present.


In [0]:
# Load the updated Delta table
final_df = spark.table("workspace.default.superstore_delta")

display(final_df.limit(10))

# print("=" * 50)
print("Validation Summary")
# print("=" * 50)

print("Original Rows :", df.count())
print("Rows After Cleaning :", clean_df.count())
print("Rows After Merge :", final_df.count())

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
138,CA-2016-145583,2016-10-13,2016-10-19,Standard Class,LC-16885,Lena Creighton,Consumer,United States,Roseville,California,95661,West,OFF-BI-10004781,Office Supplies,Binders,GBC Wire Binding Strips,76.176,3,0.2,26.6616
140,CA-2016-145583,2016-10-13,2016-10-19,Standard Class,LC-16885,Lena Creighton,Consumer,United States,Roseville,California,95661,West,FUR-FU-10001706,Furniture,Furnishings,Longer-Life Soft White Bulbs,43.12,14,0.0,20.6976
148,CA-2016-114489,2016-12-05,2016-12-09,Standard Class,JE-16165,Justin Ellison,Corporate,United States,Franklin,Wisconsin,53132,Central,TEC-PH-10000215,Technology,Phones,Plantronics Cordless�Phone Headset�with In-line Volume - M214C,384.45,11,0.0,103.8015
153,CA-2016-158834,2016-03-13,2016-03-16,First Class,TW-21025,Tamara Willingham,Home Office,United States,Scottsdale,Arizona,85254,West,TEC-PH-10001254,Technology,Phones,Jabra BIZ 2300 Duo QD Duo Corded�Headset,203.184,2,0.2,15.2388
175,US-2014-100853,2014-09-14,2014-09-19,Standard Class,JB-15400,Jennifer Braxton,Corporate,United States,Chicago,Illinois,60623,Central,OFF-AP-10000891,Office Supplies,Appliances,Kensington 7 Outlet MasterPiece HOMEOFFICE Power Control Center,52.448,2,0.8,-131.12
179,US-2015-101511,2015-11-21,2015-11-23,Second Class,JE-15745,Joel Eaton,Consumer,United States,Newark,Ohio,43055,East,OFF-SU-10002189,Office Supplies,Supplies,Acme Rosewood Handle Letter Opener,15.88,5,0.2,-3.7715
197,CA-2014-140004,2014-03-21,2014-03-25,Standard Class,CB-12025,Cassandra Brandow,Consumer,United States,Hamilton,Ohio,45011,East,OFF-AR-10004027,Office Supplies,Art,"Binney & Smith inkTank Erasable Desk Highlighter, Chisel Tip, Yellow, 12/Box",6.048,3,0.2,1.5876
206,CA-2017-108329,2017-12-09,2017-12-14,Standard Class,LE-16810,Laurel Elliston,Consumer,United States,Whittier,California,90604,West,TEC-PH-10001918,Technology,Phones,Nortel Business Series Terminal T7208 Digital phone,444.768,4,0.2,44.4768
218,CA-2016-130162,2016-10-28,2016-11-01,Standard Class,JH-15910,Jonathan Howell,Consumer,United States,Los Angeles,California,90032,West,OFF-ST-10001328,Office Supplies,Storage,"Personal Filing Tote with Lid, Black/Gray",93.06,6,0.0,26.0568
223,CA-2015-169397,2015-12-24,2015-12-27,First Class,JB-15925,Joni Blumstein,Consumer,United States,Dublin,Ohio,43017,East,FUR-FU-10000087,Furniture,Furnishings,"Executive Impressions 14"" Two-Color Numerals Wall Clock",72.704,4,0.2,19.0848


Validation Summary
Original Rows : 9994
Rows After Cleaning : 9994
Rows After Merge : 9995


In [0]:
from pyspark.sql.functions import count

duplicates = (
    final_df.groupBy("Row_ID")
            .agg(count("*").alias("Duplicate_Count"))
            .filter("Duplicate_Count > 1")
)

display(duplicates)

Row_ID,Duplicate_Count


### My Observation

The validation results confirm that the MERGE operation was successful. The final table contains the expected records, and no duplicate `Row_ID` values were found after processing.

# Assignment Summary

### Tasks Completed

- Loaded the Superstore dataset
- Explored dataset structure and schema
- Checked for missing values
- Removed duplicate records
- Renamed columns for Delta compatibility
- Created a Delta table
- Simulated incremental data
- Updated existing records
- Added a new record
- Performed MERGE operation
- Validated the final Delta table

# Challenges Faced

- While creating the Delta table, I encountered an error because some column names contained spaces.
- I resolved the issue by replacing spaces with underscores before saving the data in Delta format.
- I also learned how the MERGE operation can update existing records and insert new records in a single transaction.

# Key Learnings

- Understood the benefits of Delta Lake over traditional file formats.
- Learned how to perform incremental data processing using the MERGE operation.
- Learned the importance of data validation after updates.

# Conclusion

This assignment helped me understand how Delta Lake simplifies incremental data processing. By combining data cleaning, Delta tables, MERGE operations, and validation, I gained practical experience with an important data engineering workflow.